In [ ]:
# %%
import pandas as pd
import requests

EIA_KEY = "YOUR EIA KEY"  
START_YEAR = "2001"
END_YEAR = "2024"
STATE = "TX"

# --- 1. GENERATION DATA ---
GEN_URL = "https://api.eia.gov/v2/electricity/electric-power-operational-data/data/"
gen_params = {
    "api_key": EIA_KEY,
    "frequency": "annual",
    "data[0]": "generation",
    "facets[location][]": STATE,
    "facets[sectorid][]": "98",         
    "facets[fueltypeid][]": ["ALL", "SUN"],  
    "start": START_YEAR,
    "end": END_YEAR,
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "length": 5000 
}

gen_req = requests.get(GEN_URL, params=gen_params)
gen_df = pd.DataFrame(gen_req.json()['response']['data'])
gen_df['generation'] = pd.to_numeric(gen_df['generation'], errors='coerce')
gen_df['period'] = pd.to_datetime(gen_df['period'], format='%Y').dt.year

fuel_pivot = gen_df.pivot_table(index='period', columns='fueltypeid', values='generation', aggfunc='sum').fillna(0)
fuel_pivot['pct_solar'] = (fuel_pivot['SUN'] / fuel_pivot['ALL']) * 100
tx_generation = fuel_pivot[['pct_solar']].reset_index()

# --- 2. RETAIL DATA (Simplified for Price Only) ---
PRICE_URL = "https://api.eia.gov/v2/electricity/retail-sales/data/"
price_params = {
    "api_key": EIA_KEY,
    "frequency": "annual",
    "data[0]": "price", # Just pulling the raw cents/kWh now
    "facets[stateid][]": STATE,
    "facets[sectorid][]": "RES",  
    "start": START_YEAR,
    "end": END_YEAR,
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "length": 5000 
}

price_req = requests.get(PRICE_URL, params=price_params)
price_df = pd.DataFrame(price_req.json()['response']['data'])

price_df['price'] = pd.to_numeric(price_df['price'], errors='coerce')
price_df['period'] = pd.to_datetime(price_df['period'], format='%Y').dt.year

tx_retail = price_df[['period', 'price']].rename(
    columns={'price': 'res_price_cents_kwh'}
).copy()

# --- 3. MASTER MERGE ---
tx_final_df = pd.merge(tx_generation, tx_retail, on='period', how='inner')
tx_final_df['pct_solar'] = tx_final_df['pct_solar'].round(2)
tx_final_df['res_price_cents_kwh'] = tx_final_df['res_price_cents_kwh'].round(2)

In [ ]:
# %%
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Apply your BTT Master Style
try:
    plt.style.use('btt_style') 
except:
    pass 

fig, ax1 = plt.subplots()

color_solar = '#E6A01D' 
color_price = '#1E3A8A' # Swapped the variable name for clarity

# --- AXIS 1: Solar Percentage (Left) ---
ax1.plot(tx_final_df['period'], tx_final_df['pct_solar'], color=color_solar, label='Utility-Scale Solar')
ax1.set_ylabel("% of Texas Grid USS", color=color_solar, fontweight='bold', labelpad=15)
ax1.tick_params(axis='y', labelcolor=color_solar)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax1.set_ylim(0, 10) 

# --- AXIS 2: Retail Electricity Price (Right) ---
ax2 = ax1.twinx()
# Swapped to the new retail price column
ax2.plot(tx_final_df['period'], tx_final_df['res_price_cents_kwh'], color=color_price, label='Retail Price')

# Updated label to reflect cents per kWh
ax2.set_ylabel('Texas REP (cents/kWh)', color=color_price, fontweight='bold', labelpad=15)
ax2.tick_params(axis='y', labelcolor=color_price)

# Formatting the ticks to show the cent symbol with one decimal place
ax2.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

# Snapped the limits to hug Texas historical retail prices 
ax2.set_ylim(6, 16) 

# --- FORMATTING & CLEANUP ---
# Re-snapped the limits to perfectly hug a 24-year timeline
ax1.set_xlim(1999, 2026) 

# Generate a perfectly equidistant 4-year interval (2000, 2004, ... 2024)
custom_ticks = list(range(2001, 2026, 4))
ax1.set_xticks(custom_ticks)

# Rotate labels 45 degrees and align them perfectly to the tick marks
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')

# Dropped the pad slightly since the rotation naturally creates breathing room
ax1.tick_params(axis='x', pad=10)

# --- THE TITLE ---
# Tweaked the title to reflect the shift from Bills to Prices
#fig.suptitle("USS vs. Texans' Power Bills", fontsize=46, fontweight='bold')

# Force the outer border thickness directly on the figure object
fig.patch.set_linewidth(4) 

fig.savefig("fig1.png", dpi=600)

In [ ]:
import gridstatus


# Initialize the ERCOT ISO class
ercot = gridstatus.Ercot()

# 1. Pull historical hourly fuel mix from the EIA archive
# You can grab a free key in 30 seconds at eia.gov
eia = gridstatus.EIA(api_key="m3sfdl54PXmaMuJcludgJXHrzWJbOlwjVpRFh4Gm")

generation_df = eia.get_dataset(
    dataset="electricity/rto/fuel-type-data", # Removed the trailing /data
    start="2026-07-11",
    end="2026-07-13", 
    facets={"respondent": ["ERCO"]} 
)

# 2. Pull the Real-Time Settlement Point Prices (SPP) for yesterday
# The market MUST be explicitly defined as "REAL_TIME_15_MIN"
pricing_df = ercot.get_spp(date="2026-07-11", market="REAL_TIME_15_MIN")

# 3. Filter the pricing data for the general Hub Average
# (Otherwise you will be plotting thousands of individual neighborhood nodes)
hub_pricing_df = pricing_df[pricing_df["Location"] == "HB_HUBAVG"]

# Add this to your existing pull cell
demand_df = ercot.get_load(date="2026-07-11")

In [ ]:
# --- THE FIX: Standardize the EIA timestamp column to match ERCOT ---
generation_df = generation_df.rename(columns={'Interval Start': 'Time'})

# 1. Break the link to the master dataframe to avoid copy warnings
hub_pricing_df = hub_pricing_df.copy()

# 2. Convert both to datetime objects
generation_df['Time'] = pd.to_datetime(generation_df['Time'])
hub_pricing_df['Time'] = pd.to_datetime(hub_pricing_df['Time'])

# 3. Safely convert to Central Time ONLY if they haven't been stripped yet
if generation_df['Time'].dt.tz is not None:
    generation_df['Time'] = generation_df['Time'].dt.tz_convert('US/Central').dt.tz_localize(None)

if hub_pricing_df['Time'].dt.tz is not None:
    hub_pricing_df['Time'] = hub_pricing_df['Time'].dt.tz_convert('US/Central').dt.tz_localize(None)

# --- NEW STEP: Snap the EIA generation window perfectly to the ERCOT pricing window ---
min_time = hub_pricing_df['Time'].min()
max_time = hub_pricing_df['Time'].max()

generation_df = generation_df[
    (generation_df['Time'] >= min_time) & 
    (generation_df['Time'] <= max_time)
]

# 4. Apply the 2-hour centered moving average to smooth out the wholesale price volatility
hub_pricing_df['SPP_Smoothed'] = hub_pricing_df['SPP'].rolling(window=8, center=True).mean()

# Convert to datetime and strip timezone for demand
demand_df['Time'] = pd.to_datetime(demand_df['Time'])
if demand_df['Time'].dt.tz is not None:
    demand_df['Time'] = demand_df['Time'].dt.tz_convert('US/Central').dt.tz_localize(None)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# Apply your custom newsletter style file
plt.style.use('btt_style')

# Set up the figure and the first axis 
fig, ax1 = plt.subplots()

# Define colors
color_gen = '#E6A01D' 
color_price = '#1E3A8A' 
color_demand = '#6B7280'

# ---------------------------------------------------------
# 1. LEFT AXIS: Solar Generation 
# ---------------------------------------------------------
ax1.plot(generation_df['Time'], generation_df['Solar'], color=color_gen, linewidth=3)
ax1.set_ylabel('Texas USS Generation (MW)', color=color_gen, labelpad=15)
ax1.tick_params(axis='y', labelcolor=color_gen)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-4000,80000)

# ---------------------------------------------------------
# 2. RIGHT AXIS (Standard): Wholesale Price
# ---------------------------------------------------------
ax2 = ax1.twinx()
ax2.plot(hub_pricing_df['Time'], hub_pricing_df['SPP_Smoothed'] / 10, color=color_price, linewidth=3)
ax2.set_ylabel('Texas WEP (cents/kWh)', color=color_price, labelpad=15)
ax2.tick_params(axis='y', labelcolor=color_price)
ax2.grid(False)
ax2.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)

# ---------------------------------------------------------
# 3. RIGHT AXIS (Offset): Total Demand
# ---------------------------------------------------------
ax3 = ax1.twinx()
# Push this axis outward by 15% so it doesn't overlap the pricing axis
ax3.spines['right'].set_position(('axes', 1.15)) 

ax3.fill_between(demand_df['Time'], demand_df['Load'], color=color_demand, alpha=0.15)
ax3.set_ylabel('Total Demand (MW)', color=color_demand, labelpad=15)
ax3.tick_params(axis='y', labelcolor=color_demand)

# ---------------------------------------------------------
# Formatting & Export
# ---------------------------------------------------------
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%#I%p'))
ax1.xaxis.set_major_locator(mdates.HourLocator(interval=3))

fig.patch.set_linewidth(4) 



# bbox_inches='tight' guarantees the 3rd axis label isn't cropped in the png
fig.savefig("fig2.png", bbox_inches='tight')
plt.show()